# Compute Author Embeddings for AER

Computes unweighted mean of work embeddings per author cluster.
Used as a content-similarity signal in the AER pipeline for merge/split detection.

- **Source**: `work_embeddings_v2` (414M work embeddings, 1536-dim)
- **Method**: Unweighted mean of all work embeddings per author
- **Output**: One 1536-dim embedding per author in Databricks (not ES)
- **Job**: #105.6 aer-author-embeddings

In [ ]:
# Configuration
OUTPUT_TABLE = "openalex.aer.author_embeddings"
EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
WORKS_TABLE = "openalex.works.openalex_works"
EMBEDDING_DIM = 1536
N_BATCHES = 5  # fewer batches = fewer full table scans (was 200)

## Setup output table

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS aer")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (
        author_id BIGINT,
        work_count INT,
        embedding ARRAY<DOUBLE>
    )
""")

existing = spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']
print(f"Output table has {existing:,} authors already computed")

## Check remaining work

Count how many authors still need embeddings computed.

In [ ]:
# Fast remaining estimate — skip the expensive COUNT DISTINCT on works table
# We know there are ~73M authors with at least 1 work embedding
TOTAL_AUTHORS_ESTIMATE = 73_000_000
remaining = max(0, TOTAL_AUTHORS_ESTIMATE - existing)
print(f"Already computed: {existing:,} | Remaining (est): {remaining:,}")

if remaining == 0:
    print("Nothing to do!")

## Process batches

Batch by `author_id % N_BATCHES` — purely in Spark SQL, no driver-side collection.
Fewer batches = fewer full scans of the 414M embeddings table.
Checkpoints after each batch via INSERT INTO.

In [ ]:
import time
from datetime import datetime

def get_completed_count():
    return spark.sql(f"SELECT COUNT(*) AS n FROM {OUTPUT_TABLE}").collect()[0]['n']

def process_batch(batch_num):
    """Compute mean embeddings for authors where author_id % N_BATCHES == batch_num."""
    spark.sql(f"""
        INSERT INTO {OUTPUT_TABLE}
        WITH author_work_embeddings AS (
            SELECT
                CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
                e.embedding
            FROM {EMBEDDINGS_TABLE} e
            JOIN {WORKS_TABLE} w ON e.work_id = CAST(w.id AS STRING)
            LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
            WHERE a.author.id IS NOT NULL
              AND e.embedding IS NOT NULL
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) % {N_BATCHES} = {batch_num}
              AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) NOT IN (
                  SELECT author_id FROM {OUTPUT_TABLE}
                  WHERE author_id % {N_BATCHES} = {batch_num}
              )
        ),
        summed AS (
            SELECT
                author_id,
                CAST(COUNT(*) AS INT) AS work_count,
                aggregate(
                    collect_list(embedding),
                    cast(array_repeat(cast(0.0 AS DOUBLE), {EMBEDDING_DIM}) AS ARRAY<DOUBLE>),
                    (acc, x) -> transform(acc, (v, i) -> v + x[i])
                ) AS sum_embedding
            FROM author_work_embeddings
            GROUP BY author_id
        )
        SELECT
            author_id,
            work_count,
            transform(sum_embedding, v -> v / work_count) AS embedding
        FROM summed
    """)


start_time = time.time()
start_count = get_completed_count()

print(f"Processing {N_BATCHES} batches (modulo-based)")
print("=" * 70)

for batch_num in range(N_BATCHES):
    batch_start = time.time()
    try:
        process_batch(batch_num)
        batch_elapsed = time.time() - batch_start

        current = get_completed_count()
        added = current - start_count
        total_elapsed = time.time() - start_time
        rate = added / total_elapsed if total_elapsed > 0 else 0
        remaining_n = remaining - added if remaining > added else 0
        eta_min = remaining_n / rate / 60 if rate > 0 else 0

        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"Batch {batch_num + 1}/{N_BATCHES}: {batch_elapsed:.0f}s | "
            f"Total: {current:,} | "
            f"+{added:,} this run | "
            f"Rate: {rate:.0f}/s | "
            f"ETA: {eta_min:.0f}min"
        )
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ERROR batch {batch_num} : {e}")
        print("Waiting 60s before continuing...")
        time.sleep(60)

print("=" * 70)
final = get_completed_count()
total_time = (time.time() - start_time) / 60
print(f"Done! {final:,} total authors in {total_time:.1f}min")

## Verify results

In [ ]:
spark.sql(f"""
    SELECT
        COUNT(*) AS total_authors,
        AVG(work_count) AS avg_works_per_author,
        PERCENTILE(work_count, 0.5) AS median_works,
        PERCENTILE(work_count, 0.95) AS p95_works,
        MAX(work_count) AS max_works,
        SUM(CASE WHEN work_count = 1 THEN 1 ELSE 0 END) AS single_work_authors
    FROM {OUTPUT_TABLE}
""").show(truncate=False)

In [ ]:
# Verify all embeddings are 1536-dim
spark.sql(f"""
    SELECT SIZE(embedding) AS embedding_dim, COUNT(*) AS n
    FROM {OUTPUT_TABLE}
    GROUP BY SIZE(embedding)
""").show()

In [ ]:
# Spot check: top authors by work count with L2 norm
spark.sql(f"""
    SELECT
        author_id,
        work_count,
        ROUND(SQRT(AGGREGATE(embedding, CAST(0.0 AS DOUBLE), (acc, v) -> acc + v * v)), 4) AS l2_norm
    FROM {OUTPUT_TABLE}
    ORDER BY work_count DESC
    LIMIT 10
""").show(truncate=False)

## Optimize table

In [ ]:
spark.sql(f"OPTIMIZE {OUTPUT_TABLE}")
print(f"Table optimized: {OUTPUT_TABLE}")